# Evaluate base vs fine-tuned Nemotron-Mini-4B-Instruct

**Switch this runtime to a T4 GPU** (Runtime -> Change runtime type -> T4 GPU) before running.

Runs the same fixed prompt set through (1) the plain base model and (2) the base model with the LoRA adapter from [nemotron-rag-serving-lab](https://github.com/abdelwb/nemotron-rag-serving-lab) attached, records both sets of outputs plus latency, and writes one CSV that `analysis/gap_analysis.py` consumes afterward. No training happens here, just inference, so this is a single, simpler session than the fine-tune notebooks -- no Drive persistence needed.

In [ ]:
# Capped below transformers 5.x for the same reason as nemotron-rag-serving-lab's
# finetune notebooks: avoids a breaking change relative to what this notebook is
# written against. No trl/SFTTrainer needed here since this is inference only.
!pip install -q "transformers>=4.44,<5.0" "peft>=0.12" "accelerate>=0.33" "bitsandbytes>=0.43" pandas

In [ ]:
!git clone -q https://github.com/abdelwb/model-eval-gap-lab.git
%cd model-eval-gap-lab

In [ ]:
from huggingface_hub import login

# Only needed if nvidia/Nemotron-Mini-4B-Instruct is gated on your account --
# the adapter itself is public. A read-scoped token is enough here (no
# push_to_hub in this notebook): https://huggingface.co/settings/tokens
login()

In [ ]:
import json

with open("eval/prompts.jsonl") as f:
    prompts = [json.loads(line) for line in f if line.strip()]
print(f"Loaded {len(prompts)} prompts across {len(set(p['category'] for p in prompts))} categories")

In [ ]:
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "nvidia/Nemotron-Mini-4B-Instruct"
ADAPTER_REPO = "abdelwb/nemotron-mini-4b-daring-anteater-lora"
MAX_NEW_TOKENS = 150

# float16, not bfloat16: the free-tier T4 has no native bf16 support. This is
# inference-only (no backward pass, no GradScaler), so the dtype headaches the
# fine-tune notebooks hit don't apply here -- float16 throughout is enough.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)
base_model.eval()


def generate(model, prompt_text):
    messages = [{"role": "user", "content": prompt_text}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    start = time.perf_counter()
    with torch.no_grad():
        output_ids = model.generate(
            inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    latency_s = time.perf_counter() - start

    new_tokens = output_ids[0][inputs.shape[-1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return text.strip(), len(new_tokens), latency_s

In [ ]:
rows = []

for i, p in enumerate(prompts):
    text, n_tokens, latency_s = generate(base_model, p["prompt"])
    rows.append({
        "variant": "base",
        "id": p["id"],
        "category": p["category"],
        "prompt": p["prompt"],
        "reference": p.get("reference"),
        "expect_refusal": p.get("expect_refusal", False),
        "keywords": json.dumps(p.get("keywords", [])),
        "output": text,
        "output_tokens": n_tokens,
        "latency_s": round(latency_s, 3),
    })
    print(f"[base {i+1}/{len(prompts)}] {p['id']}")

print(f"Done: {len(rows)} base-model generations")

In [ ]:
from peft import PeftModel

finetuned_model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
finetuned_model.eval()

for i, p in enumerate(prompts):
    text, n_tokens, latency_s = generate(finetuned_model, p["prompt"])
    rows.append({
        "variant": "finetuned",
        "id": p["id"],
        "category": p["category"],
        "prompt": p["prompt"],
        "reference": p.get("reference"),
        "expect_refusal": p.get("expect_refusal", False),
        "keywords": json.dumps(p.get("keywords", [])),
        "output": text,
        "output_tokens": n_tokens,
        "latency_s": round(latency_s, 3),
    })
    print(f"[finetuned {i+1}/{len(prompts)}] {p['id']}")

print(f"Done: {len(rows)} total rows (base + finetuned)")

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
out_path = "results/eval_results.csv"
df.to_csv(out_path, index=False)
print(f"Wrote {len(df)} rows to {out_path}")
df.head()

## Getting the CSV back into the repo

This notebook doesn't push on its own (no write token needed for an eval run). Download `results/eval_results.csv` from the Colab file browser on the left, then either commit it yourself, or hand it back in your chat with Claude to commit -- same pattern used for this project's other Colab outputs.

Next steps once the CSV is real:
```bash
python analysis/gap_analysis.py
python dashboard/build_dashboard.py
```